# Decision Trees / Regressor

**Type:** Guided machine-learning practice notebook (course-based study).  
**Purpose:** Explore the dataset and the modeling/preprocessing operations illustrated in the code below.  
**Data in this folder:** `Student_Performance.csv`.  
**Methods referenced:** `DecisionTreeRegressor`.

> **Reproducibility note:** Run cells sequentially from this notebook’s folder. Outputs are historical study results, not independently validated benchmarks. The data files were supplied with the original archive; see `DATA_SOURCES.md` at the repository root for provenance checks.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Student_Performance.csv")

In [3]:
df

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,Yes,4,2,23.0
9996,7,64,Yes,8,5,58.0
9997,6,83,Yes,8,5,74.0
9998,9,97,Yes,7,0,95.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  object 
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 468.9+ KB


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
df.columns

Index(['Hours Studied', 'Previous Scores', 'Extracurricular Activities',
       'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index'],
      dtype='object')

In [7]:
df["Extracurricular Activities"] = df["Extracurricular Activities"].map({"Yes" : 1, "No" : 0})

In [8]:
df

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,1,4,2,23.0
9996,7,64,1,8,5,58.0
9997,6,83,1,8,5,74.0
9998,9,97,1,7,0,95.0


In [9]:
X = df.drop("Performance Index", axis = 1)
y = df["Performance Index"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

In [11]:
scaler = StandardScaler()

In [12]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
from sklearn.tree import DecisionTreeRegressor

In [14]:
params = {
    "criterion" : ["squared_error", "absolute_error", "poisson"],
    "splitter" : ["best", "random"],
    "max_depth" : [2, 3, 4, 5, 6, 7, 8, 9, 10, None]
}

In [15]:
from sklearn.model_selection import GridSearchCV

In [16]:
grid = GridSearchCV(estimator = DecisionTreeRegressor(), param_grid = params, scoring="neg_mean_absolute_error", cv = 5)

In [17]:
import warnings
warnings.filterwarnings("ignore")

In [18]:
grid.fit(X_train_scaled, y_train)

,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'max_depth': [2, 3, ...], 'splitter': ['best', 'random']}"
,scoring,'neg_mean_absolute_error'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'poisson'


In [19]:
grid.best_params_

{'criterion': 'poisson', 'max_depth': 9, 'splitter': 'best'}

In [20]:
y_pred = grid.predict(X_test_scaled)

In [21]:
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, r2_score

In [22]:
print(mean_absolute_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

1.9533835311897558
0.9835910063267312


In [23]:
train_pred = grid.predict(X_train_scaled)
print("Train MAE:", mean_absolute_error(y_train, train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Train MAE: 1.5899256038024492
Test MAE: 1.9533835311897558


In [24]:
feature_importance = pd.Series(
    grid.best_estimator_.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

Previous Scores                     0.840143
Hours Studied                       0.156015
Sleep Hours                         0.002108
Sample Question Papers Practiced    0.001475
Extracurricular Activities          0.000259
dtype: float64
